In [ ]:
import pandas as pd
print("done install pandas")
import json

In [ ]:
# importing dataframe 
df = pd.read_csv('PortPaint_unclean.csv')
df.head()
df


In [ ]:
df.iloc[0]
df["name"][0]
print(type(df["name"][0]))
print(df["name"][0])
df.head()

In [ ]:
#Extract from ["name"] column and add to new "Sitter" Column
df["Sitter"] = df["name"].str.extract(r'"Sitter"\s*:\s*"([^"]*)"')
#add artist column
df["Artist"] = df["name"].str.extract(
    r'"Artist"\s*:\s*"([^"]*)"|'
    r'"Attributed to"\s*:\s*"([^"]*)"|'
    r'"Attribution"\s*:\s*"([^"]*)"|'
    r'"Engraver"\s*:\s*"([^"]*)"'
).bfill(axis=1).iloc[:, 0]
#extract date from date column
df["Clean_Date"] = df["date"].str.extract(r'"Date"\s*:\s*"([^"]*)"')

import re
# extract gender from indexecd_topics
df["Gender"] = df["indexed_topics"].str.extract(r'(Men|Women)', flags=re.IGNORECASE)
print("done")
df.head()


In [ ]:
#extract occupations from topic
# if the indexed_topics column has "Occupation" in it, extract the list and put it in a new column called "Occupation"
df["Occupation"] = df["indexed_topics"].str.extract(r'(Military|President|Clergy|Medicine|Transportation|Governors|Art |Government)', flags=re.IGNORECASE)
#if Occupation is NaN, fill it with "Civilian"
df["Occupation"] = df["Occupation"].fillna("Civilian")
df.head(20)
df





In [ ]:
#Drop NAs
df = df[df["Sitter"].notna()].reset_index(drop=True)
#remove dates from end of some names
df["Sitter"] = df["Sitter"].str.split(",", n=1).str[0].str.strip()
#remove extraneous info from the end of some artists
df["Artist"] = df["Artist"].str.split(",", n = 1).str[0].str.strip()
#only keep numbers in Clean_Date
df["Clean_Date"] = df["Clean_Date"].astype(str).str.extract('(\d+)').astype(int)

df["Clean_Date"]


# df.to_csv("check_dates.csv")
# df.head()
df.head()


In [ ]:
#Export here:
df.to_csv("PortPaint_Clean.csv", index=False)


In [ ]:
#summary df
sitter_counts = df["Sitter"].value_counts().reset_index()
sitter_counts.columns = ["Sitter", "Count"]
print(sitter_counts)

In [ ]:
sitter_counts.to_csv("sitter_counts_test.csv")

Take cleaned.csv and remove sitters not used for project 2

In [ ]:
clean_before_removal = pd.read_csv("cleaned.csv")
clean_before_removal.head()

In [ ]:
sitters_to_keep = ["George Washington",
    "Benjamin Franklin",
    "Thomas Jefferson",
    "John Adams",
    "Alexander Hamilton",
    "James Madison"
]
cleaned_test = clean_before_removal[clean_before_removal["Sitter"].isin(sitters_to_keep)].reset_index(drop=True)

In [ ]:
print(cleaned_test["Sitter"].value_counts())
cleaned_test

In [ ]:
cleaned_test.to_csv("cleaned_test.csv")

In [ ]:
#testing if pandas can read the data
readable = pd.read_csv("cleaned_test.csv")
readable_from_github = pd.read_csv("https://raw.githubusercontent.com/nmolnar-parsons/major-studio-1/refs/heads/main/Project_2/Data/cleaned_test.csv")

In [ ]:
#more cleaning
portpaint_editedocc = pd.read_csv('PortPaint_editedOcc.csv')
portpaint_editedocc.head()

In [ ]:
#remove thumnails NAN
portpaint_editedocc = portpaint_editedocc[portpaint_editedocc["thumbnail"].notna()].reset_index(drop=True)
#remove sitters with "unidentified"
portpaint_editedocc = portpaint_editedocc[~portpaint_editedocc["Sitter"].str.contains("unidentified", case=False, na=False)].reset_index(drop=True)
#change Gender to title case
portpaint_editedocc["Gender"] = portpaint_editedocc["Gender"].str.title()



portpaint_editedocc

In [ ]:
#export
portpaint_editedocc.to_csv("PortPaint_edited_cleaned.csv", index=False)

Further refining of dataset - getting first and last initial

In [18]:
df_for_initials = pd.read_csv("PortPaint_withfaces.csv")
df_for_initials.head()

,collectionsURL,unitCode,dataSource,title,EDANid,guid,recordLink,lastUpdateDate,creditLine,date,...,mediaCount,mediaURLs,thumbnail,Sitter,Artist,Clean_Date,Gender,Occupation,faces,face_urls
0,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,George Washington,edanmdm:saam_XX108A,http://n2t.net/ark:/65665/vk7c06aa8a1-8da1-47c...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""1803?""}",...,1,['https://ids.si.edu/ids/deliveryService?id=SA...,https://ids.si.edu/ids/iiif/SAAM-XX108A_1/full...,George Washington,William Winstanley,1803,Men,President,"[[623, 441, 998, 968]]",https://github.com/nmolnar-parsons/revperiod_p...
1,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,The Davis Children (Eliza Cheever Davis and Jo...,edanmdm:saam_2019.6.12,http://n2t.net/ark:/65665/vk7db2de5a6-554f-456...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""1795""}",...,1,['https://ids.si.edu/ids/deliveryService?id=SA...,https://ids.si.edu/ids/iiif/SAAM-2019.6.12_1/f...,Eliza Cheever Davis\nJohn Derby Davis,Edward Savage,1795,Women,Civilian,"[[1194, 446, 1359, 682], [492, 318, 670, 546]]",https://github.com/nmolnar-parsons/revperiod_p...
2,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,John Swanwick,edanmdm:saam_2010.16.2,http://n2t.net/ark:/65665/vk73f915b19-cb1a-4db...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""ca. 1800""}",...,1,['https://ids.si.edu/ids/deliveryService?id=SA...,https://ids.si.edu/ids/iiif/SAAM-2010.16.2_1/f...,John Swanwick,Matthew Pratt,1800,Men,Civilian,"[[480, 314, 815, 796]]",https://github.com/nmolnar-parsons/revperiod_p...
3,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,Joseph Ball,edanmdm:saam_2010.16.1,http://n2t.net/ark:/65665/vk7106741b0-4bc3-416...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""1798 -1805""}",...,1,['https://ids.si.edu/ids/deliveryService?id=SA...,https://ids.si.edu/ids/iiif/SAAM-2010.16.1_1/f...,Joseph Ball,Christian Gullager,1798,Men,Civilian,"[[639, 551, 953, 1032]]",https://github.com/nmolnar-parsons/revperiod_p...
4,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,The Wiley Family,edanmdm:saam_2006.12.2,http://n2t.net/ark:/65665/vk70dfab768-2a70-4e9...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""1771""}",...,1,['https://ids.si.edu/ids/deliveryService?id=SA...,https://ids.si.edu/ids/iiif/SAAM-2006.12.2_1/f...,John Wiley,William Williams,1771,Men,Civilian,"[[1268, 496, 1332, 583], [942, 586, 1002, 666]...",https://github.com/nmolnar-parsons/revperiod_p...


In [19]:
#remove any rows with \n in Sitter
df_for_initials = df_for_initials[~df_for_initials["Sitter"].str.contains(r"\\n", na=False)].reset_index(drop=True)
df_for_initials.head()

,collectionsURL,unitCode,dataSource,title,EDANid,guid,recordLink,lastUpdateDate,creditLine,date,...,mediaCount,mediaURLs,thumbnail,Sitter,Artist,Clean_Date,Gender,Occupation,faces,face_urls
0,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,George Washington,edanmdm:saam_XX108A,http://n2t.net/ark:/65665/vk7c06aa8a1-8da1-47c...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""1803?""}",...,1,['https://ids.si.edu/ids/deliveryService?id=SA...,https://ids.si.edu/ids/iiif/SAAM-XX108A_1/full...,George Washington,William Winstanley,1803,Men,President,"[[623, 441, 998, 968]]",https://github.com/nmolnar-parsons/revperiod_p...
1,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,John Swanwick,edanmdm:saam_2010.16.2,http://n2t.net/ark:/65665/vk73f915b19-cb1a-4db...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""ca. 1800""}",...,1,['https://ids.si.edu/ids/deliveryService?id=SA...,https://ids.si.edu/ids/iiif/SAAM-2010.16.2_1/f...,John Swanwick,Matthew Pratt,1800,Men,Civilian,"[[480, 314, 815, 796]]",https://github.com/nmolnar-parsons/revperiod_p...
2,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,Joseph Ball,edanmdm:saam_2010.16.1,http://n2t.net/ark:/65665/vk7106741b0-4bc3-416...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""1798 -1805""}",...,1,['https://ids.si.edu/ids/deliveryService?id=SA...,https://ids.si.edu/ids/iiif/SAAM-2010.16.1_1/f...,Joseph Ball,Christian Gullager,1798,Men,Civilian,"[[639, 551, 953, 1032]]",https://github.com/nmolnar-parsons/revperiod_p...
3,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,The Wiley Family,edanmdm:saam_2006.12.2,http://n2t.net/ark:/65665/vk70dfab768-2a70-4e9...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""1771""}",...,1,['https://ids.si.edu/ids/deliveryService?id=SA...,https://ids.si.edu/ids/iiif/SAAM-2006.12.2_1/f...,John Wiley,William Williams,1771,Men,Civilian,"[[1268, 496, 1332, 583], [942, 586, 1002, 666]...",https://github.com/nmolnar-parsons/revperiod_p...
4,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,Robert Hooper,edanmdm:saam_2006.12.1,http://n2t.net/ark:/65665/vk72d0f38a8-a827-4f8...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""ca. 1770-1772""}",...,1,['https://ids.si.edu/ids/deliveryService?id=SA...,https://ids.si.edu/ids/iiif/SAAM-2006.12.1_1/f...,Jr. Robert Hooper,John Singleton Copley,1770,Men,Civilian,"[[608, 338, 832, 623]]",https://github.com/nmolnar-parsons/revperiod_p...


In [20]:
df_for_initials["first_initial"] = df_for_initials["Sitter"].str.split().str[0].str[0]
df_for_initials["last_initial"] = df_for_initials["Sitter"].str.split().str[-1].str[0]
df_for_initials.head(20)


,collectionsURL,unitCode,dataSource,title,EDANid,guid,recordLink,lastUpdateDate,creditLine,date,...,thumbnail,Sitter,Artist,Clean_Date,Gender,Occupation,faces,face_urls,first_initial,last_initial
0,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,George Washington,edanmdm:saam_XX108A,http://n2t.net/ark:/65665/vk7c06aa8a1-8da1-47c...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""1803?""}",...,https://ids.si.edu/ids/iiif/SAAM-XX108A_1/full...,George Washington,William Winstanley,1803,Men,President,"[[623, 441, 998, 968]]",https://github.com/nmolnar-parsons/revperiod_p...,G,W
1,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,John Swanwick,edanmdm:saam_2010.16.2,http://n2t.net/ark:/65665/vk73f915b19-cb1a-4db...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""ca. 1800""}",...,https://ids.si.edu/ids/iiif/SAAM-2010.16.2_1/f...,John Swanwick,Matthew Pratt,1800,Men,Civilian,"[[480, 314, 815, 796]]",https://github.com/nmolnar-parsons/revperiod_p...,J,S
2,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,Joseph Ball,edanmdm:saam_2010.16.1,http://n2t.net/ark:/65665/vk7106741b0-4bc3-416...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""1798 -1805""}",...,https://ids.si.edu/ids/iiif/SAAM-2010.16.1_1/f...,Joseph Ball,Christian Gullager,1798,Men,Civilian,"[[639, 551, 953, 1032]]",https://github.com/nmolnar-parsons/revperiod_p...,J,B
3,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,The Wiley Family,edanmdm:saam_2006.12.2,http://n2t.net/ark:/65665/vk70dfab768-2a70-4e9...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""1771""}",...,https://ids.si.edu/ids/iiif/SAAM-2006.12.2_1/f...,John Wiley,William Williams,1771,Men,Civilian,"[[1268, 496, 1332, 583], [942, 586, 1002, 666]...",https://github.com/nmolnar-parsons/revperiod_p...,J,W
4,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,Robert Hooper,edanmdm:saam_2006.12.1,http://n2t.net/ark:/65665/vk72d0f38a8-a827-4f8...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""ca. 1770-1772""}",...,https://ids.si.edu/ids/iiif/SAAM-2006.12.1_1/f...,Jr. Robert Hooper,John Singleton Copley,1770,Men,Civilian,"[[608, 338, 832, 623]]",https://github.com/nmolnar-parsons/revperiod_p...,J,H
5,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,Col. Nathaniel Darby,edanmdm:saam_1999.87.7A,http://n2t.net/ark:/65665/vk7230f9012-ccc2-4d3...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""ca. 1798""}",...,https://ids.si.edu/ids/iiif/SAAM-1999.87.7A_1/...,Col. Nathaniel Darby,Unidentified,1798,Men,Military,"[[605, 620, 979, 1145]]",https://github.com/nmolnar-parsons/revperiod_p...,C,D
6,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,"Louis-Guillaume Otto, comte de Mosloy",edanmdm:saam_1999.87.6,http://n2t.net/ark:/65665/vk74c7d73d1-a7cf-409...,https://americanart.si.edu/collections/search/...,2024-11-22,"{""Credit Line"": ""Smithsonian American Art Muse...","{""Date"": ""ca. 1780""}",...,https://ids.si.edu/ids/iiif/SAAM-1999.87.6_1/f...,Comte de Mosloy Louis Guillaume Otto,Charles Willson Peale,1780,Men,Civilian,"[[1043, 806, 1400, 1320]]",https://github.com/nmolnar-parsons/revperiod_p...,C,O
7,https://collections.si.edu/search/detail/edanm...,SAAM,Smithsonian American Art Museum,William Shippen,edanmdm:saam_1999.87.5,http://n2t.net/ark:/65665/vk7c98492ae-0664-47b...,https://am

In [21]:
#save output
df_for_initials.to_csv("PortPaint_Use.csv", index=False)